In [ ]:
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO

# --- PATH SETUP ---
video_input_path = r"C:\Users\srava\OneDrive\FindingLanes\input_data\road_car_view.mp4"
base_path = r"C:\Users\srava\OneDrive\FindingLanes"
output_folder = os.path.join(base_path, "output_data")

os.makedirs(output_folder, exist_ok=True)
video_output_path = os.path.join(output_folder, "final_high_contrast_lanes.mp4")

class SimulatorView:
    def __init__(self):
        # Lane History
        self.left_fit_history = []
        self.right_fit_history = []
        self.history_length = 15 
        self.road_texture = None 
        
        # Data Smoothing
        self.rad_history = []
        self.off_history = []
        
        # Physics
        self.prev_frame_time = 0
        self.new_frame_time = 0
        self.simulated_speed = 60 
        
        # Model
        print("Loading YOLOv8 Model for Vehicle Detection...")
        self.model = YOLO("yolov8n.pt") 
        self.vehicle_classes = [2, 3, 5, 7] 

        self.vehicle_widths = {
            2: 1.8, 3: 0.8, 5: 2.5, 7: 2.5
        }

    def generate_road_texture(self, width, height):
        base_color = np.full((height, width, 3), (40, 40, 40), dtype=np.uint8)
        noise = np.random.randint(0, 50, (height, width, 3)).astype(np.uint8)
        texture = cv2.add(base_color, noise)
        return texture

    def measure_curvature_real(self, left_fit, right_fit, h):
        ym_per_pix = 30/720 
        xm_per_pix = 3.7/700 
        ploty = np.linspace(0, h-1, h)
        left_fit_cr = np.polyfit(ploty*ym_per_pix, (left_fit[0]*ploty**2 + left_fit[1]*ploty + left_fit[2])*xm_per_pix, 2)
        right_fit_cr = np.polyfit(ploty*ym_per_pix, (right_fit[0]*ploty**2 + right_fit[1]*ploty + right_fit[2])*xm_per_pix, 2)
        y_eval = np.max(ploty)
        left_curverad = ((1 + (2*left_fit_cr[0]*y_eval*ym_per_pix + left_fit_cr[1])**2)**1.5) / np.absolute(2*left_fit_cr[0])
        right_curverad = ((1 + (2*right_fit_cr[0]*y_eval*ym_per_pix + right_fit_cr[1])**2)**1.5) / np.absolute(2*right_fit_cr[0])
        car_position = 1280 / 2 
        lane_center_position = (left_fit[0]*h**2 + left_fit[1]*h + left_fit[2] + right_fit[0]*h**2 + right_fit[1]*h + right_fit[2]) / 2
        offset_meters = (car_position - lane_center_position) * xm_per_pix
        return (left_curverad + right_curverad) / 2, offset_meters

    def draw_steering_wheel(self, img, angle, x, y, radius):
        wheel_img = np.zeros((radius*2, radius*2, 3), dtype=np.uint8)
        cv2.circle(wheel_img, (radius, radius), radius-5, (200, 200, 200), 8, cv2.LINE_AA)
        cv2.circle(wheel_img, (radius, radius), radius-5, (50, 50, 50), 2, cv2.LINE_AA)
        cv2.line(wheel_img, (radius, radius), (radius, 5), (180, 180, 180), 6, cv2.LINE_AA) 
        cv2.line(wheel_img, (radius, radius), (5, radius+20), (180, 180, 180), 6, cv2.LINE_AA) 
        cv2.line(wheel_img, (radius, radius), (radius*2-5, radius+20), (180, 180, 180), 6, cv2.LINE_AA) 
        M = cv2.getRotationMatrix2D((radius, radius), angle, 1)
        rotated_wheel = cv2.warpAffine(wheel_img, M, (radius*2, radius*2))
        h_w, w_w = rotated_wheel.shape[:2]
        roi = img[y:y+h_w, x:x+w_w]
        img2gray = cv2.cvtColor(rotated_wheel, cv2.COLOR_BGR2GRAY)
        ret, mask = cv2.threshold(img2gray, 10, 255, cv2.THRESH_BINARY)
        mask_inv = cv2.bitwise_not(mask)
        img_bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
        wheel_fg = cv2.bitwise_and(rotated_wheel, rotated_wheel, mask=mask)
        dst = cv2.add(img_bg, wheel_fg)
        img[y:y+h_w, x:x+w_w] = dst

    def draw_realistic_car(self, img, x, y, w, h):
        car_color = (190, 190, 200); cabin_color = (30, 30, 40); glass_color = (70, 70, 80)
        overlay = img.copy()
        cv2.ellipse(overlay, (int(x+w/2+10), int(y+h/2+15)), (int(w*1.2)//2, int(h*1.15)//2), 0, 0, 360, (20,20,20), -1)
        cv2.addWeighted(overlay, 0.5, img, 0.5, 0, img)
        ww = int(w*0.18); wh = int(h*0.25)
        for wx, wy in [(x-5, y+int(h*0.15)), (x+w-ww+5, y+int(h*0.15)), (x-5, y+int(h*0.7)), (x+w-ww+5, y+int(h*0.7))]:
            cv2.rectangle(img, (wx, wy), (wx+ww, wy+wh), (20,20,20), -1)
        body_pts = np.array([[x+w*0.15, y], [x+w*0.85, y], [x+w, y+h*0.25], [x+w, y+h*0.85], [x+w*0.8, y+h], [x+w*0.2, y+h], [x, y+h*0.85], [x, y+h*0.25]], np.int32)
        cv2.fillPoly(img, [body_pts], car_color)
        cabin_w = int(w*0.75); cabin_h = int(h*0.5)
        cx = int(x + (w-cabin_w)/2); cy = int(y + h*0.2)
        cv2.fillPoly(img, [np.array([[cx+10, cy], [cx+cabin_w-10, cy], [cx+cabin_w, cy+cabin_h], [cx, cy+cabin_h]], np.int32)], cabin_color)
        cv2.fillPoly(img, [np.array([[cx+15, cy+5], [cx+cabin_w-15, cy+5], [cx+cabin_w, cy+cabin_h*0.3], [cx, cy+cabin_h*0.3]], np.int32)], glass_color)
        cv2.circle(img, (int(x+w*0.15), int(y+h*0.05)), 8, (255,255,255), -1, cv2.LINE_AA)
        cv2.circle(img, (int(x+w*0.85), int(y+h*0.05)), 8, (255,255,255), -1, cv2.LINE_AA)
        cv2.fillPoly(img, [np.array([[x+w*0.1, y+h], [x+w*0.3, y+h], [x+w*0.3, y+h-10], [x+w*0.1, y+h-10]], np.int32)], (0,0,200))
        cv2.fillPoly(img, [np.array([[x+w*0.9, y+h], [x+w*0.7, y+h], [x+w*0.7, y+h-10], [x+w*0.9, y+h-10]], np.int32)], (0,0,200))

    # --- UPDATED: HIGHLIGHTING LOGIC ---
    def thresholding(self, img):
        # 1. LAB for Yellow - Strict Thresholds
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        b_channel = lab[:,:,2] 
        yellow_binary = np.zeros_like(b_channel)
        # Increased threshold to avoid noise, capturing only bright yellow
        yellow_binary[b_channel > 150] = 1 

        # 2. HLS for White - Strict Thresholds
        hls = cv2.cvtColor(img, cv2.COLOR_BGR2HLS)
        l_channel = hls[:,:,1] 
        s_channel = hls[:,:,2]
        
        # CLAHE for better contrast
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        l_channel = clahe.apply(l_channel)

        white_binary = np.zeros_like(l_channel)
        # Combined lightness and saturation check for white
        white_binary[(l_channel > 210) & (s_channel < 100)] = 1 

        # 3. Combine
        combined_binary = np.zeros_like(yellow_binary)
        combined_binary[(yellow_binary == 1) | (white_binary == 1)] = 1
        
        # 4. Morphological Closing (Fills gaps in lines)
        kernel = np.ones((5,5), np.uint8)
        combined_binary = cv2.morphologyEx(combined_binary, cv2.MORPH_CLOSE, kernel)
        
        return combined_binary

    def perspective_warp(self, img, width, height):
        # Slightly wider view to catch lanes better
        src = np.float32([[width*0.40, height*0.65], [width*0.60, height*0.65], [width*0.05, height], [width*0.95, height]])
        dst = np.float32([[width*0.25, 0], [width*0.75, 0], [width*0.25, height], [width*0.75, height]])
        M = cv2.getPerspectiveTransform(src, dst)
        Minv = cv2.getPerspectiveTransform(dst, src)
        warped = cv2.warpPerspective(img, M, (width, height), flags=cv2.INTER_LINEAR)
        return warped, Minv

    def find_lanes(self, binary_warped):
        histogram = np.sum(binary_warped[binary_warped.shape[0]//2:,:], axis=0)
        
        if np.max(histogram) < 5: 
             return None, None

        midpoint = np.int64(histogram.shape[0]//2)
        leftx_current = np.argmax(histogram[:midpoint])
        rightx_current = np.argmax(histogram[midpoint:]) + midpoint
        
        nwindows = 9
        window_height = np.int64(binary_warped.shape[0]//nwindows)
        margin = 80 # Tighter margin for precision
        minpix = 50
        nonzero = binary_warped.nonzero()
        nonzeroy = np.array(nonzero[0])
        nonzerox = np.array(nonzero[1])
        left_lane_inds = []
        right_lane_inds = []
        
        for window in range(nwindows):
            win_y_low = binary_warped.shape[0] - (window+1)*window_height
            win_y_high = binary_warped.shape[0] - window*window_height
            win_xleft_low = leftx_current - margin
            win_xleft_high = leftx_current + margin
            win_xright_low = rightx_current - margin
            win_xright_high = rightx_current + margin
            
            good_left_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & (nonzerox >= win_xleft_low) & (nonzerox < win_xleft_high)).nonzero()[0]
            good_right_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) & (nonzerox >= win_xright_low) & (nonzerox < win_xright_high)).nonzero()[0]
            
            left_lane_inds.append(good_left_inds)
            right_lane_inds.append(good_right_inds)
            
            if len(good_left_inds) > minpix: leftx_current = np.int64(np.mean(nonzerox[good_left_inds]))
            if len(good_right_inds) > minpix: rightx_current = np.int64(np.mean(nonzerox[good_right_inds]))
            
        left_lane_inds = np.concatenate(left_lane_inds)
        right_lane_inds = np.concatenate(right_lane_inds)
        
        leftx = nonzerox[left_lane_inds]
        lefty = nonzeroy[left_lane_inds] 
        rightx = nonzerox[right_lane_inds]
        righty = nonzeroy[right_lane_inds]
        
        if len(leftx) > 0 and len(rightx) > 0:
            left_fit = np.polyfit(lefty, leftx, 2)
            right_fit = np.polyfit(righty, rightx, 2)
            return left_fit, right_fit
        return None, None

    def process_split_view(self, frame):
        height, width = frame.shape[:2]
        if self.road_texture is None:
            self.road_texture = self.generate_road_texture(width, height)

        self.new_frame_time = time.time()
        fps_val = 0
        if self.prev_frame_time > 0:
            fps_val = 1 / (self.new_frame_time - self.prev_frame_time)
        self.prev_frame_time = self.new_frame_time
        fps_val = int(fps_val)

        # --- STEP 1: OBJECT DETECTION ---
        results = self.model(frame, stream=True, verbose=False)
        detection_view = np.copy(frame)
        vehicle_count = 0
        min_distance = 1000 
        
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                
                if cls in self.vehicle_classes and conf > 0.4: # Increased confidence
                    vehicle_count += 1
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    box_w = x2 - x1
                    real_width_meters = self.vehicle_widths.get(cls, 1.8) 
                    
                    if box_w > 0:
                        distance_m = (real_width_meters * 600) / box_w 
                    else:
                        distance_m = 100

                    if distance_m < min_distance:
                        min_distance = distance_m

                    box_color = (0, 255, 0)
                    if distance_m < 15: box_color = (0, 165, 255)
                    if distance_m < 8: box_color = (0, 0, 255)

                    cv2.rectangle(detection_view, (x1, y1), (x2, y2), box_color, 2)
                    label = f"{self.model.names[cls]} {distance_m:.1f}m"
                    cv2.putText(detection_view, label, (x1, y1 - 5), 0, 0.5, box_color, 2, cv2.LINE_AA)

        # --- STEP 2: LANE HIGHLIGHTING ---
        binary = self.thresholding(frame)
        warped, Minv = self.perspective_warp(binary, width, height)
        left_fit, right_fit = self.find_lanes(warped)
        
        current_rad = 0
        current_off = 0
            
        if left_fit is not None and right_fit is not None:
            self.left_fit_history.append(left_fit)
            self.right_fit_history.append(right_fit)
            if len(self.left_fit_history) > self.history_length:
                self.left_fit_history.pop(0)
                self.right_fit_history.pop(0)
        
        if len(self.left_fit_history) > 0:
            smooth_left = np.average(self.left_fit_history, axis=0)
            smooth_right = np.average(self.right_fit_history, axis=0)
            
            current_rad, current_off = self.measure_curvature_real(smooth_left, smooth_right, height)
            
            self.rad_history.append(current_rad)
            self.off_history.append(current_off)
            if len(self.rad_history) > 15: self.rad_history.pop(0); self.off_history.pop(0)
            avg_rad = np.mean(self.rad_history)
            avg_off = np.mean(self.off_history)
        else:
            smooth_left, smooth_right = None, None
            avg_rad = 1000 
            avg_off = 0

        target_speed = 100 if avg_rad > 800 else 40
        if min_distance < 8: target_speed = 0
            
        if self.simulated_speed < target_speed: self.simulated_speed += 0.5 
        elif self.simulated_speed > target_speed: self.simulated_speed -= 1.0

        left_view = detection_view 
        warning_color = (0, 255, 0) 
        warning_text = "STATUS: SAFE"
        
        if min_distance < 8:
            warning_text = "BRAKE! COLLISION AHEAD"
            warning_color = (0, 0, 255)
            overlay_flash = left_view.copy()
            cv2.rectangle(overlay_flash, (0, 0), (width, height), (0, 0, 255), -1)
            cv2.addWeighted(overlay_flash, 0.4, left_view, 0.6, 0, left_view)
        elif abs(current_off) > 0.6: 
            warning_text = "WARNING: LANE DEPARTURE"
            warning_color = (0, 140, 255)
            
        if smooth_left is not None:
            ploty = np.linspace(0, height-1, height)
            left_fitx = smooth_left[0]*ploty**2 + smooth_left[1]*ploty + smooth_left[2]
            right_fitx = smooth_right[0]*ploty**2 + smooth_right[1]*ploty + smooth_right[2]
            center_fitx = (left_fitx + right_fitx) / 2
            
            # --- HIGHLIGHTING: BRIGHT GREEN LANE ---
            warp_zero = np.zeros_like(warped).astype(np.uint8)
            color_warp = np.dstack((warp_zero, warp_zero, warp_zero))
            
            # Create points for polygon
            pts_left = np.array([np.transpose(np.vstack([left_fitx, ploty]))])
            pts_right = np.array([np.flipud(np.transpose(np.vstack([right_fitx, ploty])))])
            pts = np.hstack((pts_left, pts_right))
            
            # Fill Polygon with Bright Green
            cv2.fillPoly(color_warp, np.int_([pts]), (0, 255, 0))
            
            # Draw Thick Red Lines on the Edges for Contrast
            cv2.polylines(color_warp, np.int_([pts_left]), False, (0, 0, 255), 15) 
            cv2.polylines(color_warp, np.int_([pts_right]), False, (0, 0, 255), 15)
            
            newwarp = cv2.warpPerspective(color_warp, Minv, (width, height))
            
            # Blend: Increased opacity (0.4) to make it shine more
            left_view = cv2.addWeighted(left_view, 1, newwarp, 0.4, 0)

            # --- RIGHT VIEW ---
            right_view = self.road_texture.copy()
            pts_left_d = np.array([np.transpose(np.vstack([left_fitx, ploty]))], np.int32)
            pts_right_d = np.array([np.transpose(np.vstack([right_fitx, ploty]))], np.int32)
            
            cv2.polylines(right_view, pts_left_d, False, (50, 50, 255), 10, cv2.LINE_AA)
            cv2.polylines(right_view, pts_right_d, False, (50, 255, 50), 10, cv2.LINE_AA)
            pts_center = np.array([np.transpose(np.vstack([center_fitx, ploty]))], np.int32)
            cv2.polylines(right_view, pts_center, False, (255, 255, 0), 4, cv2.LINE_AA)

            car_w, car_h = 130, 240
            car_x = int(width / 2) - car_w // 2
            car_y = height - car_h - 40
            self.draw_realistic_car(right_view, car_x, car_y, car_w, car_h)
            
            # --- DASHBOARD ---
            cv2.rectangle(left_view, (15, 15), (380, 270), (0, 0, 0), -1) 
            cv2.rectangle(left_view, (15, 15), (380, 270), (255, 255, 255), 2)
            
            font = cv2.FONT_HERSHEY_SIMPLEX
            cv2.putText(left_view, f"FPS: {fps_val}", (30, 50), font, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(left_view, f"Speed: {int(self.simulated_speed)} km/h", (30, 90), font, 0.8, (0, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(left_view, f"Radius: {int(avg_rad)}m", (30, 130), font, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
            
            off_dir = "Right" if avg_off > 0 else "Left"
            cv2.putText(left_view, f"Offset: {abs(avg_off):.2f}m {off_dir}", (30, 170), font, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(left_view, warning_text, (30, 210), font, 0.7, warning_color, 2, cv2.LINE_AA)
            cv2.putText(left_view, f"Vehicles: {vehicle_count}", (30, 250), font, 0.7, (0, 140, 255), 2, cv2.LINE_AA)

            steer_ang = 0
            if avg_rad < 2000:
                steer_ang = (1000 / avg_rad) * 40 
                if left_fitx[-1] < left_fitx[0]: steer_ang = steer_ang
                else: steer_ang = -steer_ang
            
            self.draw_steering_wheel(left_view, steer_ang, width-150, height-150, 60)
            cv2.putText(left_view, "STEERING", (width-140, height-20), font, 0.6, (255,255,255), 2, cv2.LINE_AA)

        else:
            right_view = self.road_texture.copy()

        combined_final = np.hstack((left_view, right_view))
        header = np.zeros((50, combined_final.shape[1], 3), dtype=np.uint8)
        cv2.putText(header, "AI SELF-DRIVING SYSTEM (High Contrast Lanes)", (350, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
        final_output = np.vstack((header, combined_final))
        
        return final_output

# --- RUN ---
sim = SimulatorView()
cap = cv2.VideoCapture(video_input_path)

if not cap.isOpened():
    print(f"Error: Video not found at {video_input_path}")
else:
    print(f"Generating HIGH CONTRAST View... Saving to: {video_output_path}")
    w = int(cap.get(3))
    h = int(cap.get(4))
    out = cv2.VideoWriter(video_output_path, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (w * 2, h + 50))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        try:
            final_output = sim.process_split_view(frame)
            display_output = cv2.resize(final_output, (1280, 400)) 
            cv2.imshow("Safety Plus AI View", display_output)
            out.write(final_output)
        except Exception as e:
            print(f"Error: {e}")
            pass
        if cv2.waitKey(1) == ord('q'): break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Done!")

Loading YOLOv8 Model for Vehicle Detection...
Generating HIGH CONTRAST View... Saving to: C:\Users\srava\OneDrive\FindingLanes\output_data\final_high_contrast_lanes.mp4


C:\Users\srava\AppData\Local\Temp\ipykernel_8340\3796037669.py:186: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit(lefty, leftx, 2)
C:\Users\srava\AppData\Local\Temp\ipykernel_8340\3796037669.py:186: RankWarning: Polyfit may be poorly conditioned
  left_fit = np.polyfit(lefty, leftx, 2)
